### Preparation

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

#### Q1. How many lesson pages ?

In [67]:
print(len(files))

72


### Q2. Indexing and searching

In [68]:
from minsearch import Index

def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index

index=build_index(documents)

In [69]:
query= "How does the agentic loop keep calling the model until it stops?"
search_results = index.search(query)
print(search_results[0]['filename'])

01-agentic-rag/lessons/14-agentic-loop.md


### Q3. RAG

In [ ]:
query="How does the agentic loop keep calling the model until it stops?"

In [16]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
openai_client = OpenAI()

In [ ]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the lessons.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

answer = assistant.rag(query)
print(answer[0])
print(f"\n Token Used:{answer[1]}")

The loop keeps calling the model by using a `while True` loop and checking whether the model returned any `function_call` items.

- It sends the current `messages` to the model.
- If the response contains a `function_call`, it runs the tool, appends the tool result to `messages`, and keeps looping.
- If there are no function calls in that turn, it breaks out of the loop.

So the exit condition is simple: **no function calls this turn means the model is done, and the loop stops**.

 Token Used:7020


### Q4. Chunking

In [38]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

295


## Q5. RAG with chunking

In [39]:
chunk_index=build_index(chunks)

In [66]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the lessons.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=chunk_index,
    llm_client=openai_client,
    instructions=instructions,
)

answer = assistant.rag(query)
print(answer[0])
print(f"\n Token Used:{answer[1]}")

print(f"{int(7020/2204)} times fewer chunks")

The loop keeps calling the model with a `while True` loop, and after each turn it checks whether the model made any `function_call`s.

- If there was at least one function call, the code runs the tool, appends the tool output, and loops again.
- If there were no function calls this turn, it breaks out of the loop.

So the stop condition is: **no function calls in the model’s response**.

 Token Used:2204
3 times fewer chunks


## Q6. Turning it into an agent

In [57]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [58]:
def search(query: str) -> dict[str, str]:
    """
    Search the lessons for entries matching the given query.
    """
    return chunk_index.search(
        query,
        num_results=5
    )

In [59]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [60]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the lessons for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [61]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [62]:
instructions="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [63]:
result = runner.loop(
    prompt='How does the agentic loop work, and how is it different from plain RAG?',
    callback=callback,
)

-> Response received


-> Response received
